## EDA for LUNA25 dataset
This notebook aims at exploring the dataset (image and annotations) and create train-test-val splits.

In [10]:
from pathlib import Path
import os
import pandas as pd
from typing import List

base_path = Path("/opt/data/LUNA25")
labels = base_path / "luna25_nodule_blocks" / "image"
metadata = base_path / "luna25_nodule_blocks" / "metadata"
images = base_path / "luna25_images"


In [ ]:
def get_df(path: Path, columns: List=None, split: str="_", extension: str=".npy"):
    # list all .npy files under the labels directory (recursively)
    npy_files = [f.stem for f in sorted(path.rglob(f"*{extension}"))]

    if columns is None:
        n_col = len(npy_files[0].split(split))
        columns = [f"Col_{n}" for n in range(n_col)]

    pat_dict = [dict([(k,v) for k,v in zip(columns, p.split(split))]) for p in npy_files]

    df = pd.DataFrame(pat_dict)

    return df


In [23]:
import pandas as pd

df = pd.read_csv(base_path / "annotations.csv")

In [20]:
import numpy as np

test_npy_path = [x for x in metadata.iterdir() if x.is_file()]
test_npy = np.load(test_npy_path[0], allow_pickle=True)



In [22]:
test_npy

array({'origin': array([-154.3999939 , -173.80859375,  -70.60546875]), 'spacing': array([2.       , 0.5859375, 0.5859375]), 'transform': array([[1., 0., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])}, dtype=object)

### Generate train-test split
The following code creates a balanced (i.e. stratified) train-test split in which the class proportion remains unchanged in both splits

In [3]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# Params
csv_path = Path("luna25-annotations.csv")
random_state = 42

# Load the CSV file
df = pd.read_csv(csv_path)

# Display class distribution
print("Original class distribution:")
print(df['label'].value_counts())
print(f"\nTotal samples: {len(df)}")


Original class distribution:
0    5608
1     555
Name: label, dtype: int64

Total samples: 6163


In [4]:
# Create stratified train-test split (80-20 by default)
train_df, test_df = train_test_split(
    df, 
    test_size=0.2,  # 20% for test, 80% for train
    stratify=df['label'],  # This ensures balanced class distribution
    random_state= random_state  # For reproducibility
)

# Verify the split
print("\n" + "="*50)
print("TRAIN SET:")
print(f"Total samples: {len(train_df)}")
print(train_df['label'].value_counts())
print(f"Class proportions:\n{train_df['label'].value_counts(normalize=True)}")

print("\n" + "="*50)
print("TEST SET:")
print(f"Total samples: {len(test_df)}")
print(test_df['label'].value_counts())
print(f"Class proportions:\n{test_df['label'].value_counts(normalize=True)}")



TRAIN SET:
Total samples: 4930
0    4486
1     444
Name: label, dtype: int64
Class proportions:
0    0.909939
1    0.090061
Name: label, dtype: float64

TEST SET:
Total samples: 1233
0    1122
1     111
Name: label, dtype: int64
Class proportions:
0    0.909976
1    0.090024
Name: label, dtype: float64


In [ ]:
# Save the splits to separate CSV files
train_df.to_csv('luna25-train.csv', index=False)
test_df.to_csv('luna25-test.csv', index=False)

print("\n" + "="*50)
print("✓ Files saved:")
print("  - luna25-train.csv")
print("  - luna25-test.csv")